In [ ]:
import os
try:
    path_initialized
except NameError:
    path_initialized = True
    os.chdir('..')

import numpy as np
from sympy.abc import x, y

# from qldpc import codes
import networkx as nx
import matplotlib.pyplot as plt
import stim
import sinter

import src.device as device
import src.plotting as plotter
from src.RotatedSurfaceCode import RotatedSurfaceCode
from src.HGPCode import HGPCode
from src.QECCode import TestCode

In [ ]:
d = 5
hwp = device.hardware_params(1e-3)
dev = device.UnitCellDevice(d+2, d+2, hwp)
code = RotatedSurfaceCode(d)

# Want to optimize ancilla qubit schedules from the middle out, to ensure
# schedules mesh well together
cx,cy = tuple(np.mean(code.qubit_coords, axis=0))
anc_order = []
anc_queue = set(code.X_ancilla_indices)
while anc_queue:
    next_anc = min(anc_queue, key=lambda anc: np.linalg.norm([code.qubit_coords[anc][0]-cx, code.qubit_coords[anc][1]-cy]))
    anc_order.append(next_anc)
    anc_queue.remove(next_anc)
anc_queue = set(code.Z_ancilla_indices)
while anc_queue:
    next_anc = min(anc_queue, key=lambda anc: np.linalg.norm([code.qubit_coords[anc][0]-cx, code.qubit_coords[anc][1]-cy]))
    anc_order.append(next_anc)
    anc_queue.remove(next_anc)

data_coords = {q:((code.qubit_coords[q][0]+1)//2, (code.qubit_coords[q][1]+1)//2) for q in code.data_indices}
assert len(set(data_coords.values())) == len(data_coords)
sched = dev.compile_QEC_schedule(
    code,
    data_coords,
    code.check_cx_layers,
    rounds=1,
    use_highways=True,
    refocus_shuttle_noise=False,
    optimize_ancilla_start=False,
    separate_X_Z=True
)

In [ ]:
[code.qubit_coords[d] for d in code.X_ancilla_indices]

In [ ]:
data_init_coords = [((x+1)//2, (y+1)//2) for (x,y) in [code.qubit_coords[q] for q in code.data_indices]]
anc_init_coords = [((x+2)//2, (y+2)//2) for (x,y) in [code.qubit_coords[q] for q in code.X_ancilla_indices + code.Z_ancilla_indices]]

for i,d in enumerate(code.data_indices):
    plt.text(*data_init_coords[i], d)
plt.xlim(0, code.dx+2)
plt.ylim(0, code.dx+2)
plt.show()
for i,a in enumerate(code.X_ancilla_indices + code.Z_ancilla_indices):
    plt.text(*anc_init_coords[i], a)
plt.xlim(0, code.dx+2)
plt.ylim(0, code.dx+2)
plt.show()

In [ ]:
code.z_ancilla[code.Z_ancilla_indices.index(28)]

In [ ]:
code.x_ancilla[code.X_ancilla_indices.index(34)]

In [ ]:
anc_order

In [ ]:
for instr,t in zip(sched.instructions, sched.instruction_start_times):
    print(t, instr)

In [ ]:
frames = sched.get_frames(20)
print(len(frames))
plotter.plot_transition(dev, code, frames[0:100])
plt.savefig('test.pdf')
plt.show()
plotter.plot_transition(dev, code, frames[100:150])
plt.show()

In [ ]:
sched_hand = code.get_schedule(hwp)
print(sched_hand.total_duration(), sched.total_duration())

In [ ]:
circ = sched.to_stim_circuit(code, 'Z', hwp)
print(circ.without_noise())

In [ ]:
print(circ.to_crumble_url())

sinter.collect(
    num_workers=6,
    tasks=[sinter.Task(
        circuit=circ,
    )],
    max_shots=10**6,
    max_errors=500,
    decoders='pymatching'
)

In [ ]:
print(code.get_stim('Z', 1, 1e-3, 1e-3))

In [ ]:
print(code.get_stim('Z', 1, 1e-3, 1e-3).to_crumble_url())

In [ ]:
print(circ)

In [ ]:
code.X_ancilla_indices

In [ ]:
circ = sched_hand.to_stim_circuit(code, 'Z', hwp)

print(circ.to_crumble_url())

sinter.collect(
    num_workers=6,
    tasks=[sinter.Task(
        circuit=circ,
    )],
    max_shots=10**6,
    max_errors=500,
    decoders='pymatching'
)

In [ ]:
frames = sched_hand.get_frames(10)
# plotter.plot_device_snapshot(dev, code, frames[50])
anim = plotter.animate_device(dev, code, frames, 'anim_surface_hand.gif', fps=30)

In [ ]:
frames = sched.get_frames(10)
# plotter.plot_device_snapshot(dev, code, frames[50])
anim = plotter.animate_device(dev, code, frames, 'anim_surface.gif', fps=30)